In [1]:
!pip install langchain chromadb faiss-cpu openai langchain-groq langchain-huggingface tiktoken langchain-community wikipedia

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 119.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━

In [2]:
from langchain_community.retrievers import WikipediaRetriever

retriever = WikipediaRetriever(
    top_k_results=2, lang="en"
)

In [3]:
query = "Virat kohli"

docs = retriever.invoke(query)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [4]:
print(len(docs))
for i, doc in enumerate(docs):
    print(f"{i + 1}.", doc.page_content)

NameError: name 'docs' is not defined

In [38]:
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation"
)

huggingface_model = ChatHuggingFace(llm=llm)



In [5]:
from langchain_huggingface import HuggingFaceEmbeddings


embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/home/rahulchaudhari/AI/Learning Python/Langchain/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1469.55it/s]


In [6]:
from langchain_core.documents import Document

documents = [
    Document(page_content="LangChain is a framework for building applications powered by language models."),
    Document(page_content="Vector stores are used to store and search vector embeddings."),
    Document(page_content="Embeddings represent text as numerical vectors."),
    Document(page_content="Semantic search finds documents based on meaning rather than exact keywords."),
    Document(page_content="Chroma is a popular vector database used with LangChain."),
    Document(page_content="FAISS is a library for efficient similarity search."),
    Document(page_content="A retriever fetches relevant documents from a vector store."),
    Document(page_content="Similarity search returns documents that are semantically similar to a query."),
    Document(page_content="Text splitters divide large documents into smaller chunks."),
    Document(page_content="Metadata stores additional information about a document."),
]

documents_with_metadata = [
    Document(
        page_content="Python is a popular programming language.",
        metadata={"topic": "programming", "language": "python"}
    ),
    Document(
        page_content="JavaScript is commonly used for web development.",
        metadata={"topic": "programming", "language": "javascript"}
    ),
    Document(
        page_content="Chroma is a vector database.",
        metadata={"topic": "vector_store", "technology": "chroma"}
    ),
]

In [13]:
from langchain_community.vectorstores import Chroma
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="my_chroma_collection"
)

In [15]:
# Convert vectorstore into a retriever

retriever = vector_store.as_retriever(search_kwargs={"k":2})

In [19]:
result = retriever.invoke("What is Chroma used for ")
print(len(result))
result

2


[Document(metadata={'topic': 'vector_store', 'technology': 'chroma'}, page_content='Chroma is a vector database.'),
 Document(metadata={}, page_content='Chroma is a popular vector database used with LangChain.')]

### Maximal Marginal Relevance (MMR) 

In [20]:
document_for_MMR = [
    Document(page_content="Python is a popular programming language used for web development, data science, automation, and artificial intelligence."),
    Document(page_content="Python is widely used in data science because it provides libraries such as NumPy, Pandas, and Matplotlib."),
    Document(page_content="Python is commonly used for machine learning with libraries such as Scikit-learn, PyTorch, and TensorFlow."),
    Document(page_content="Python can be used to build web applications with frameworks such as Django and Flask."),
    
    Document(page_content="JavaScript is a programming language mainly used to create interactive web applications."),
    Document(page_content="JavaScript can run in browsers and is commonly used for frontend web development."),
    Document(page_content="Node.js allows JavaScript to run outside the browser and is often used for backend development."),
    
    Document(page_content="React is a JavaScript library for building user interfaces and web applications."),
    Document(page_content="React applications are built using reusable components that manage and display UI elements."),
    Document(page_content="React uses a virtual DOM to efficiently update parts of the user interface."),
    
    Document(page_content="Vector stores are databases designed to store and search vector embeddings."),
    Document(page_content="Vector databases enable semantic search by finding vectors that are similar to a query embedding."),
    Document(page_content="Chroma is a vector store that can be used to store embeddings and perform similarity searches."),
    
    Document(page_content="RAG combines document retrieval with language models to generate answers using external information."),
    Document(page_content="Retrieval-Augmented Generation retrieves relevant documents before sending them to a language model."),
    Document(page_content="RAG systems commonly use embeddings and vector stores to retrieve relevant information."),
]

In [22]:
from langchain_community.vectorstores import FAISS

faiss_vector_store = FAISS.from_documents(
    documents=document_for_MMR,
    embedding=embedding_model
)

In [ ]:
faiss_vector_store_retriever = faiss_vector_store.as_retriever(
    search_kwargs={"k":4, "lambda_mult": 0.1}, #k = top results , lambda_mult= relevance-diversity balance, values lies b/w 0 and 1
    search_type='mmr' # <----- This enables MMR
    )

In [31]:
faiss_result = faiss_vector_store_retriever.invoke("Tell me about react")
faiss_result

[Document(id='152618ec-3a64-4f10-996b-ea775e4bcaec', metadata={}, page_content='React is a JavaScript library for building user interfaces and web applications.'),
 Document(id='466462d1-a97e-47eb-a212-d08ef68520ab', metadata={}, page_content='Retrieval-Augmented Generation retrieves relevant documents before sending them to a language model.'),
 Document(id='69725a31-b77b-4d07-958a-2c5d1048628a', metadata={}, page_content='Python is widely used in data science because it provides libraries such as NumPy, Pandas, and Matplotlib.'),
 Document(id='bc6e1eba-5f80-4f3a-bc66-a10df33ea438', metadata={}, page_content='Vector stores are databases designed to store and search vector embeddings.')]

### Multi Query Retriver

In [32]:
documents_for_MQR =  [
    Document(
        page_content="Python is a high-level programming language known for its simple and readable syntax."
    ),
    Document(
        page_content="Python is popular because developers can write programs with fewer lines of code compared with many other languages."
    ),
    Document(
        page_content="Python is widely used for data science, machine learning, automation, and artificial intelligence."
    ),
    Document(
        page_content="Python provides libraries such as NumPy, Pandas, and Matplotlib for data analysis and visualization."
    ),
    Document(
        page_content="Python is commonly used to build web applications with frameworks such as Django and Flask."
    ),
    Document(
        page_content="Python can automate repetitive tasks such as file processing, web scraping, and system administration."
    ),
    Document(
        page_content="JavaScript is primarily used for creating interactive web applications and browser-based experiences."
    ),
    Document(
        page_content="Java is widely used for enterprise applications, backend systems, and Android development."
    ),
    Document(
        page_content="Machine learning allows computers to learn patterns from data and make predictions without being explicitly programmed for every case."
    ),
    Document(
        page_content="Artificial intelligence includes techniques that enable computers to perform tasks that normally require human intelligence."
    ),
]

In [33]:
from langchain_classic.retrievers import MultiQueryRetriever

In [34]:
vector_store_multi_query_retriever = FAISS.from_documents(
    embedding=embedding_model,
    documents=documents_for_MQR
)

In [36]:
similarity_retriever = vector_store_multi_query_retriever.as_retriever(
    search_type="similarity",
    search_kwargs= {"k" : 5}
)

In [39]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vector_store_multi_query_retriever.as_retriever(
        search_kwargs={"k":5}
    ),
    llm=huggingface_model
)

In [40]:
query = "What are the advantages of Python?"
query2 = "Why is Python popular?"
query3 = "What can Python be used for?"

In [48]:
# Retrieve results

similarity_results = similarity_retriever.invoke(query3)
multiquery_results = multiquery_retriever.invoke(query3)

In [49]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Similarity Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- MultiQuery Result {i+1}---")
    print(doc.page_content)


--- Similarity Result 1 ---
Python is widely used for data science, machine learning, automation, and artificial intelligence.

--- Similarity Result 2 ---
Python is a high-level programming language known for its simple and readable syntax.

--- Similarity Result 3 ---
Python is popular because developers can write programs with fewer lines of code compared with many other languages.

--- Similarity Result 4 ---
Python is commonly used to build web applications with frameworks such as Django and Flask.

--- Similarity Result 5 ---
Python can automate repetitive tasks such as file processing, web scraping, and system administration.
******************************************************************************************************************************************************

--- MultiQuery Result 1---
Python is popular because developers can write programs with fewer lines of code compared with many other languages.

--- MultiQuery Result 2---
Python is commonly used to build w

### Contextual Compression reteriver

In [51]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor


In [52]:
documents_for_Contextual_compression = [
    Document(
        page_content="""
        Python is a popular high-level programming language.
        It was created by Guido van Rossum and first released in 1991.
        Python has a simple and readable syntax.
        Python is used for web development, data science, machine learning,
        artificial intelligence, automation, and scripting.
        Python has a large ecosystem of libraries and frameworks.
        """
    ),

    Document(
        page_content="""
        JavaScript is a programming language commonly used for web development.
        It was originally created for browsers.
        JavaScript can be used to create interactive web pages.
        Node.js allows JavaScript to run on the server.
        Popular JavaScript frameworks include React, Angular, and Vue.
        """
    ),

    Document(
        page_content="""
        Python provides many libraries for data science.
        NumPy is used for numerical computing.
        Pandas is used for data manipulation and analysis.
        Matplotlib is used for data visualization.
        Scikit-learn provides tools for machine learning.
        Python is also commonly used with TensorFlow and PyTorch.
        """
    ),

    Document(
        page_content="""
        Python can be used for web development.
        Django is a Python web framework for building web applications.
        Flask is another lightweight Python web framework.
        FastAPI is commonly used for building APIs with Python.
        Python web frameworks provide routing, request handling,
        authentication, and database integration.
        """
    ),
]

In [53]:
vector_store_for_contextual_retriver = FAISS.from_documents(
    embedding=embedding_model,
    documents=documents_for_Contextual_compression
)

In [54]:
base_retriever= vector_store_for_contextual_retriver.as_retriever(
    search_kwargs = {"k":3}
)

In [55]:

compressor = LLMChainExtractor.from_llm(huggingface_model)

In [56]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

In [57]:
query = "What is Python used for?"

In [58]:
result = compression_retriever.invoke(query)
result

[Document(metadata={}, page_content='Extracted relevant parts:\nPython is used for web development, data science, machine learning,\nartificial intelligence, automation, and scripting.'),
 Document(metadata={}, page_content='Extracted relevant parts:\n> Python provides many libraries for data science.\n> Scikit-learn provides tools for machine learning.\n> Python is also commonly used with TensorFlow and PyTorch.'),
 Document(metadata={}, page_content='Python can be used for web development.\nDjango is a Python web framework for building web applications.\nFlask is another lightweight Python web framework.\nFastAPI is commonly used for building APIs with Python.\nPython web frameworks provide routing, request handling, \nauthentication, and database integration.')]